# MR2a — Address Store MapReduce
**Input**: `manhattan_liquor_stores_geocoded.csv`
**Output**: `address_store_mr.csv`

In [3]:
import h3
import pandas as pd
import numpy as np
from collections import defaultdict
from datetime import datetime

INPUT  = 'manhattan_liquor_stores_geocoded.csv'
OUTPUT = 'address_store_mr.csv'
H3_RES = 10

df = pd.read_csv(INPUT)
df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df = df.dropna(subset=['latitude','longitude'])

print(f'Input rows: {len(df):,}')
df.head().transpose()

Input rows: 6,925


,0,1,2,3,4
serial_number,1257248,1273203,1320487,1337942,1330731
method_of_operation,RESTAURANT SERVING BEER CIDER & WINE ONLY,VESSEL SERVING LIQUOR BEER CIDER AND WINE,NaN,"RESTAURANT SERVING BEER, WINE, LIQUOR AND CIDER",IMPORTER
license_type_code,RW,VL,AX,OP,IL
county,NEW YORK,NEW YORK,NEW YORK,NEW YORK,NEW YORK
premise_name,SOLO PIZZA INC,HORNBLOWER CRUISES AND EVENTS LLC,581 R & V FOOD CORP,HSE ENTERPRISES LLC,TAT AMERICA LLC
dba,SOLO PIZZA,MANHATTAN ELITE,NaN,THE PARLOUR ROOM,HOUSE OF BURGUNDY WINES AND SPIRITS
premise_address,27 AVENUE B,PIER 62 CHELSEA PIERS,581 W 207TH ST,70 W 36TH ST,318 W 53RD ST
premise_city,NEW YORK,NEW YORK,NEW YORK,NEW YORK,NEW YORK
certificate_number,784148,803625,935179,890693,935461
license_expiration_date,2023-10-31T00:00:00.000,2023-10-31T00:00:00.000,2023-10-31T00:00:00.000,2023-10-31T00:00:00.000,2023-12-31T00:00:00.000


In [ ]:
# ── MAP ───────────────────────────────────────────────────────────────────────

def get_license_status(exp_raw):
    try:
        exp = pd.to_datetime(exp_raw)
        return 'Active' if exp >= pd.Timestamp.today() else 'License Outdated'
    except:
        return 'Unknown'

def mapper(row):
    try:
        zone_id = h3.latlng_to_cell(float(row['latitude']), float(row['longitude']), H3_RES)
    except:
        return None, None
    return zone_id, get_license_status(row.get('license_expiration_date', ''))

mapped = [mapper(row) for _, row in df.iterrows()]
mapped = [(k,v) for k,v in mapped if k is not None]
print(f'Mapped pairs: {len(mapped):,}')

In [ ]:
# ── REDUCE ────────────────────────────────────────────────────────────────────
acc = defaultdict(lambda: {'store_count':0,'active':0,'outdated':0})

for zone_id, status in mapped:
    a = acc[zone_id]
    a['store_count'] += 1
    if status == 'Active':            a['active']   += 1
    elif status == 'License Outdated': a['outdated'] += 1

rows = []
for zone_id, a in acc.items():
    rows.append({
        'zone_id':          zone_id,
        'store_count':      a['store_count'],
        'active_licenses':  a['active'],
        'outdated_licenses':a['outdated'],
    })

out = pd.DataFrame(rows).sort_values('zone_id').reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f'Output rows : {len(out):,}')
print(f'Active zones: {(out["active_licenses"]>0).sum():,}')
print(f'Saved -> {OUTPUT}')
out.head()